In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load all datasets
orders = pd.read_csv('orders_clean.csv')
order_items = pd.read_csv('order_items_clean.csv')
monthly_rev = pd.read_csv('monthly_revenue.csv')
category_rev = pd.read_csv('category_revenue.csv')
payments = pd.read_csv('payment_analysis.csv')


In [ ]:
# Quick check
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Monthly Revenue:", monthly_rev.shape)
print("Categories:", category_rev.shape)
print("Payments:", payments.shape)

In [ ]:
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)

In [ ]:
# Merge orders with order items
df = pd.merge(order_items, orders, on='order_id', how='inner')

In [ ]:
# Convert date column
df['order_date'] = pd.to_datetime(df['order_date'])


In [ ]:
# Add month and year columns
df['month'] = df['order_date'].dt.to_period('M')
df['year'] = df['order_date'].dt.year


In [ ]:
# Basic stats
print("Total Revenue: R$", round(df['price'].sum(), 2))
print("Total Orders:", df['order_id'].nunique())
print("Total Customers:", df['customer_id'].nunique())
print("Average Order Value: R$", round(df['price'].mean(), 2))
print("Date Range:", df['order_date'].min(), "to", df['order_date'].max())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
fig, ax = plt.subplots()
ax.bar(['a','b','c'], [1,2,3])
plt.show()

In [ ]:
# Build RFM table
snapshot_date = df['order_date'].max()

rfm = df.groupby('customer_id').agg(
    recency   = ('order_date', lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_id', 'nunique'),
    monetary  = ('price', 'sum')
).reset_index()

In [ ]:
# Score each dimension 1-5
rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5])
rfm['RFM_total'] = rfm[['R_score','F_score','M_score']].astype(int).sum(axis=1)


In [ ]:
# Segment labels
def segment(score):
    if score >= 13:   return 'Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 7:  return 'At Risk'
    elif score >= 4:  return 'Hibernating'
    else:             return 'Lost'

rfm['Segment'] = rfm['RFM_total'].apply(segment)

print(rfm['Segment'].value_counts())
print("\nAverage monetary by segment:")
print(rfm.groupby('Segment')['monetary'].mean().round(2).sort_values(ascending=False))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('RFM Customer Segmentation', fontsize=14, fontweight='bold')

seg_counts = rfm['Segment'].value_counts()
colors = ['#2E86AB','#A23B72','#F18F01','#C73E1D','#3B1F2B']
axes[0].bar(seg_counts.index, seg_counts.values, color=colors)
axes[0].set_title('Number of Customers per Segment')
axes[0].set_ylabel('Customers')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(seg_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

seg_money = rfm.groupby('Segment')['monetary'].mean().sort_values(ascending=False)
axes[1].barh(seg_money.index, seg_money.values, color='#A23B72')
axes[1].set_title('Average Revenue per Customer by Segment')
axes[1].set_xlabel('Avg Revenue (R$)')
for i, v in enumerate(seg_money.values):
    axes[1].text(v + 1, i, f'R${v:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('rfm_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


In [ ]:
# Prepare features
features = rfm[['recency', 'monetary']].copy()
scaler = StandardScaler()
scaled = scaler.fit_transform(features)


In [ ]:
# Elbow method
inertia = []
K = range(2, 9)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled)
    inertia.append(km.inertia_)


In [ ]:
# Fit with k=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(scaled)

In [ ]:
# Cluster summary
cluster_summary = rfm.groupby('Cluster').agg(
    Avg_Recency=('recency', 'mean'),
    Avg_Monetary=('monetary', 'mean'),
    Count=('customer_id', 'count')
).round(1)
print(cluster_summary)


In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K, inertia, marker='o', color='#2E86AB', linewidth=2)
axes[0].axvline(x=4, color='gray', linestyle='--', alpha=0.7)
axes[0].set_title('Elbow Curve - Optimal K')
axes[0].set_xlabel('Number of Clusters')
axes[0].set_ylabel('Inertia')

colors = ['#2E86AB','#A23B72','#F18F01','#C73E1D']
for i in range(4):
    mask = rfm['Cluster'] == i
    axes[1].scatter(rfm[mask]['recency'], rfm[mask]['monetary'],
                   c=colors[i], label=f'Cluster {i}', alpha=0.5, s=10)
axes[1].set_title('Customer Clusters (Recency vs Monetary)')
axes[1].set_xlabel('Recency (days)')
axes[1].set_ylabel('Monetary (R$)')
axes[1].legend()

plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save final outputs
rfm.to_csv('rfm_final.csv', index=False)

print("Columns in rfm_final:")
print(rfm.columns.tolist())
print("\nSample:")
print(rfm.head())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


In [ ]:
# Chart 1
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Brazilian E-Commerce Analysis (Olist)', fontsize=16, fontweight='bold')

monthly_rev_sorted = monthly_rev.sort_values('month')
axes[0,0].plot(monthly_rev_sorted['month'], monthly_rev_sorted['total_revenue'],
               marker='o', color='#2E86AB', linewidth=2)
axes[0,0].set_title('Monthly Revenue Trend')
axes[0,0].set_xlabel('Month')
axes[0,0].set_ylabel('Revenue (R$)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(alpha=0.3)

top10 = category_rev.head(10)
axes[0,1].barh(top10['category'], top10['total_revenue'], color='#A23B72')
axes[0,1].set_title('Top 10 Product Categories by Revenue')
axes[0,1].set_xlabel('Revenue (R$)')
axes[0,1].invert_yaxis()

axes[1,0].pie(payments['total_transactions'],
              labels=payments['payment_type'],
              autopct='%1.1f%%',
              colors=['#2E86AB','#A23B72','#F18F01','#C73E1D','#3B1F2B'])
axes[1,0].set_title('Payment Type Distribution')

axes[1,1].bar(payments['payment_type'], payments['total_value'], color='#F18F01')
axes[1,1].set_title('Total Revenue by Payment Type')
axes[1,1].set_xlabel('Payment Type')
axes[1,1].set_ylabel('Revenue (R$)')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('ecommerce_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 2
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('RFM Customer Segmentation', fontsize=14, fontweight='bold')

seg_counts = rfm['Segment'].value_counts()
colors = ['#2E86AB','#A23B72','#F18F01','#C73E1D','#3B1F2B']
axes[0].bar(seg_counts.index, seg_counts.values, color=colors)
axes[0].set_title('Number of Customers per Segment')
axes[0].set_ylabel('Customers')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(seg_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

seg_money = rfm.groupby('Segment')['monetary'].mean().sort_values(ascending=False)
axes[1].barh(seg_money.index, seg_money.values, color='#A23B72')
axes[1].set_title('Average Revenue per Customer by Segment')
axes[1].set_xlabel('Avg Revenue (R$)')
for i, v in enumerate(seg_money.values):
    axes[1].text(v + 1, i, f'R${v:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('rfm_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 3
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = rfm[['recency', 'monetary']].copy()
scaler = StandardScaler()
scaled = scaler.fit_transform(features)

inertia = []
K = range(2, 9)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled)
    inertia.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K, inertia, marker='o', color='#2E86AB', linewidth=2)
axes[0].axvline(x=4, color='gray', linestyle='--', alpha=0.7)
axes[0].set_title('Elbow Curve - Optimal K')
axes[0].set_xlabel('Number of Clusters')
axes[0].set_ylabel('Inertia')

cluster_colors = ['#2E86AB','#A23B72','#F18F01','#C73E1D']
for i in range(4):
    mask = rfm['Cluster'] == i
    axes[1].scatter(rfm[mask]['recency'], rfm[mask]['monetary'],
                   c=cluster_colors[i], label=f'Cluster {i}', alpha=0.5, s=10)
axes[1].set_title('Customer Clusters (Recency vs Monetary)')
axes[1].set_xlabel('Recency (days)')
axes[1].set_ylabel('Monetary (R$)')
axes[1].legend()

plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print("All 3 charts saved.")
